# Generate TCR candidates for a pHLA target

Companion to https://tcrchallenge.org/tutorials/tcr-design/. Read that guide for input definitions and interpretation. Run the cells in order in the same kernel.

Complete the setup guide first. Run from the DecoderTCR checkout pinned to `3e3d9889d26d79635f674940ddab039c4dc6f9f9`, with the 300M checkpoint downloaded. No model inference has been run in this notebook.

For repertoire analysis supply your own `filtered_contig_annotations.csv` and `peptide_targets.csv`. For design workflows inspect the public example and replace it with your chosen context. Record the model, source revision, device, files and settings. Generated sequences are candidates, not validated binders. Source and adapted code: MIT license.


In [ ]:
import DecoderTCR as dt
import pandas as pd

clone = {
    "trav": "TRAV21", "traj": "TRAJ6", "cdr3a": "CAVRPGGAGPFFVVF",
    "trbv": "TRBV7-9", "trbj": "TRBJ2-7", "cdr3b": "CASSLGQAYEQYF",
    "hla": "HLA-B*27:05",
}
clone["peptide"] = "LRVMMLAPF"  # Public example target; not a validated label.
# The beta-chain CDR3 changes; alpha, V/J genes, HLA and peptide stay fixed.
designs = dt.iegr(
    clone, region="cdr3b", n=3, temperature=0.1,
    seed=42, gibbs_rounds=2, gibbs_subset_size=5,
    from_genes=True, model="DecoderTCR-ESMC_300M", device="cpu",
)
designs.to_csv("cdr3b_designs.csv", index=False)
print(designs)

In [ ]:
variants = []
for i, row in designs.iterrows():
    variant = dict(clone)
    variant["name"] = f"cdr3b_candidate_{i}"
    variant["cdr3b"] = row["sequence"]
    variants.append(variant)
if not variants:
    raise ValueError("IEGR returned no candidates.")
scored = dt.score_from_components(
    variants, model="DecoderTCR-ESMC_300M", device="cpu"
)
scored.to_csv("cdr3b_candidate_scores.csv", index=False)
print(scored[["name", "cdr3b", "ok", "tcr_reason", "hla_reason",
              "pll_DecoderTCR-ESMC_300M"]])

In [ ]:
import json
from pathlib import Path
run_record = {
    "source_revision": "3e3d9889d26d79635f674940ddab039c4dc6f9f9",
    "model": "DecoderTCR-ESMC_300M", "device": "cpu",
    "guide": "https://tcrchallenge.org/tutorials/tcr-design/", "notebook": "tcr-design",
    "note": "Save this notebook with the actual inputs and generation settings used."
}
Path("tcr-design-run.json").write_text(json.dumps(run_record, indent=2))